In [1]:
import numpy as np
import sys
sys.path.append("../../")
sys.path.append("../../Visualization/")

In [2]:
sys.path.append("../../../")

In [3]:
import sys; sys.path.append('..')
import MeshFEM, mesh, sparse_matrices, benchmark, field_sampler, mesh_utilities
import inflatables_parametrization as parametrization, numpy as np, importlib, pickle, wall_generation
import utils
import py_newton_optimizer
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf

target_surf = mesh.Mesh("../../../../examples/cashew.obj")
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=True))
target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [4]:
# Choose reasonable stretching bounds in terms of the relative fusing curve widths.
alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(2, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
print(alphaMin, alphaMax)

In [5]:
# Run some iterations of the local-global algorithm to ensure a good separation between singular values.
# This step can also be used as a prediction of the feasiblity of a design surface:
# if it is unable to nearly satisfy the singular value constraints,
# the surface is probably infeasible.
lg = parametrization.LocalGlobalParametrizer(target_surf, parametrization.lscm(target_surf))

lg.alphaMin = 1.4
lg.alphaMax = np.pi / 2
print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [6]:
visualization.visualize(lg)

In [7]:
rparam = parametrization.RegularizedParametrizerSVD(target_surf, lg.uv())
rparam.alphaMin = alphaMin
rparam.alphaMax = alphaMax

In [8]:
def optimize_rparam(param, alphaRegW, phiRegW, bendRegW):
    param.alphaRegW = alphaRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    opts = NewtonOptimizerOptions()
    opts.niter = 2000
    opts.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
    #opts.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()
    cr = parametrization.regularized_parametrization_newton(param, param.rigidMotionPinVars, opts)

In [10]:
# Rerunning this cell a couple times can improve the results
benchmark.reset()
with suppress_stdout(): optimize_rparam(rparam, 100.0, 10.0, 500.0)
with suppress_stdout(): optimize_rparam(rparam, 10.0, 1.0, 250.0)
with suppress_stdout(): optimize_rparam(rparam, 1.0, 0.1, 125.0)
with suppress_stdout(): optimize_rparam(rparam, 0.1, 0.01, 62.5)
with suppress_stdout(): optimize_rparam(rparam, 0.1, 0.01, 31.25)
benchmark.report()

In [9]:
importlib.reload(utils)

In [10]:
# Report the values and gradients of each objective term
print(f'Energies: {utils.allEnergies(rparam)}')
print(f'Gradient Norms: {utils.allGradientNorms(rparam)}')

In [11]:
# Visualize the flattening
visualization.visualize(rparam)

In [12]:
importlib.reload(visualization)

In [13]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False, width = 10, height = 10)

## Upsampling and channel generation

In [14]:
import time

In [15]:
nsubdiv=4
start_time = time.time()
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)
upsampledStretches = np.clip(upsampledStretches, alphaMin, alphaMax)
(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_stripe_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles,
                                                                    wwf.canonicalWallWidthForStretchFactor(upsampledStretches), frequency=0.6)
print("computing sdf takes: ", time.time() - start_time)

In [16]:
import pickle, mesh, wall_generation, visualization, numpy as np

In [17]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, height=12)

In [18]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=4.0,
                                              minContourLen=10)

In [19]:
visualization.plot_line_segments(pts, edges, width=10, height=16)

## Meshing and inflation simulation

In [20]:
import sheet_meshing, inflation

In [21]:
import importlib
importlib.reload(sheet_meshing)

In [22]:
m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, pts, edges, triArea=2)

In [23]:
isheet = inflation.InflatableSheet(m, iwv)
isheet.setRelaxedStiffnessEpsilon(1e-6)
uv = rparam.uv()

In [24]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [25]:
# Manually stretch the sheet onto the target surface by applying the inverse of the parametrization
paramSampler = field_sampler.FieldSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [26]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-4
opts.niter = 1000

In [27]:
# Fix the boundary positions
import boundaries
bdryVars = boundaries.getBoundaryVars(isheet)
fixedVars = bdryVars

In [28]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [29]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = bdryVars, 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [30]:
isheet.pressure = 1e-5

In [31]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [32]:
isheet.pressure = 0.03

In [33]:
opts.niter = 1000

In [34]:
import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [35]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);

## Shape Optimization

In [36]:
# Reset the inflation and set up target-attraction forces
isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)
targetAttractedSheet = inflation.TargetAttractedInflation(isheet, target_surf)
targetAttractedSheet.energy(targetAttractedSheet.EnergyType.Fitting)

In [37]:
targetAttractedSheet.targetSurfaceFitter().holdClosestPointsFixed = True
targetAttractedSheet.fittingWeight = 1e-5

In [38]:
# Re-inflate, this time applying target-attraction forces.
import time
isheet.pressure = 0.025


benchmark.reset()
cr = inflation.inflation_newton(targetAttractedSheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

In [39]:
# Set up the sheet optimizer
import sheet_optimizer, opt_config
origDesignMesh = isheet.mesh().copy()

sheet_opt = sheet_optimizer.PySheetOptimizer(targetAttractedSheet, fixedVars, renderMode=sheet_optimizer.RenderMode.PYTHREEJS,
                                             detActivationThreshold=0.9, detActivationThresholdTubeTri=0.5,
                                             originalDesignMesh=origDesignMesh, fusingCurveSmoothnessConfig=opt_config.FusingCurveSmoothnessParams(0.0, 0.0, 1.0, 1.0))

In [40]:
# Configure some more weights
sheet_opt.rso.compressionPenaltyWeight = 1e-6
fcs = sheet_opt.rso.fusingCurveSmoothness()
fcs.interiorWeight = 0.05

In [41]:
sheet_opt.flat_viewer.showWireframe()
sheet_opt.viewer()

In [43]:
sheet_opt.deploy_viewer.getCameraParams()

In [109]:
sheet_opt.setSolver(sheet_optimizer.Solver.SCIPY, 200)
sheet_opt.rso.getEquilibriumSolver().options.niter = 20

In [ ]:
# Run the optimization
sheet_opt.setSolver(sheet_optimizer.Solver.SCIPY)
sheet_opt.optimize()

In [ ]:
# Lower the interior weight
fcs = sheet_opt.rso.fusingCurveSmoothness()
fcs.interiorWeight = 0.05

In [ ]:
# Continue the optimization
sheet_opt.optimize()

In [ ]:
utils.allGradientNorms(sheet_opt.rso)

In [ ]:
utils.allEnergies(sheet_opt.rso)

In [ ]:
# Remove the target-attraction force and recompute the equilibrium
targetAttractedSheet.fittingWeight = 1e-8
inflation.inflation_newton(targetAttractedSheet, sheet_opt.rso.fixedEquilibriumVars(), sheet_opt.opts)
viewer.update()

In [ ]:
# # Save the full state for later reloading with `sheet_optimizer.load()`
# sheet_opt.save('20240116_igloo_optimized_sheet_opt.pkl.gz')

### Generate Fabrication Files

In [ ]:
sheet_opt.save('20240119_cashew_optimized_sheet_opt.pkl.gz')

In [ ]:
scaleFactor = 1 # Factor for fine-tuning size to fit the machine's build area
channelMargin = 0
tabMargin = 0

In [ ]:
isheet = sheet_opt.rso.sheet()
optMesh = sheet_opt.rso.mesh().copy()
origMesh = sheet_opt.rso.originalMesh().copy()
import inflation
tas = sheet_opt.rso.targetAttractedInflation()
tsf = tas.targetSurfaceFitter()
targetSurf = mesh.Mesh(tsf.targetSurfaceV, tsf.targetSurfaceF)
iwv = [isheet.isWallVtx(i) for i in range(isheet.mesh().numVertices())]

In [ ]:
import fabrication
importlib.reload(fabrication)
fabrication.writeFabricationData('fabrication_data/cashew/parallel_tube', origMesh, optMesh, iwv, targetSurf, uv,
                                 scale=scaleFactor, numTabs=20, inletOffset=0, tabOffset=0.6 / 20,
                                 channelMargin=channelMargin, tabMargin=tabMargin, tabWidth=5, tabHeight=8, fuseSeamWidth=0.01, inletScale=0,
                                 overlap=0.0, smartOuterChannel=True)